# AlphaEarth / GEE — Extraction d'échantillons pour embeddings

Ce notebook montre comment utiliser `AlphaEarthClient` pour échantillonner des pixels depuis Google Earth Engine et préparer des vecteurs prêts pour un modèle foundation.
**Important** : n'exécutez pas `ee.Initialize()` sans avoir configuré vos credentials locaux (service account ou `earthengine authenticate`).

In [ ]:
# Imports et vérifications
import sys
print('Python', sys.version)
try:
    import ee, numpy as np, matplotlib.pyplot as plt
    from geocongoai.ia.alphaearth import AlphaEarthClient
    print('Imports OK: ee, numpy, geocongoai')
except Exception as e:
    print('Imports manquants ou warning:', e)
    # installer: pip install earthengine-api

In [ ]:
# Authentification (exemple): utiliser un compte de service avec key.json
client = AlphaEarthClient()
try:
    client.authenticate()
    print('GEE initialized')
except Exception as e:
    print('GEE init failed (interactive or credentials missing):', e)
    print('You can still run a mocked sampling for demo purposes')

In [ ]:
# Exemple: créer une géométrie ee et échantillonner (si auth ok)
try:
    import ee
    geom = ee.Geometry.Point([29.5, -2.55]).buffer(1000)
    res = client.extract_embeddings(geometry=geom, start_date='2022-01-01', end_date='2022-12-31')
    if 'error' in res:
        print('Sampling error:', res['error'])
    else:
        print('Samples count:', res.get('count'))
        # afficher un extrait
        samples = res.get('samples') or []
        if samples:
            import numpy as np
            arr = np.array(samples)
            print('Samples shape:', arr.shape)
except Exception as e:
    print('GEE sampling skipped (no auth):', e)
    # Demo: créer des vecteurs synthétiques
    import numpy as np
    samples = (np.random.rand(100, 4) * 1000).tolist()
    print('Created', len(samples), 'synthetic samples for demo')

In [ ]:
# Sauvegarder les échantillons localement pour ingestion par un modèle
import json, os
out = {'samples': samples}
os.makedirs('examples/data', exist_ok=True)
with open('examples/data/alphaearth_samples.json', 'w') as f:
    json.dump(out, f)
print('Saved samples to examples/data/alphaearth_samples.json')

## Remarques finales
- Pour de grandes zones, utilisez `ee.batch.Export` (export to GCS) au lieu de `getInfo()`
- Ne committez jamais de fichiers de credentials dans le repo; utilisez les secrets ou fichiers locaux non suivis.